# FIT File Parsing Guide for Node.js Payload Shape

This notebook explains how a `.fit` file is structured and how to transform it into the payload format used by your backend (`analysisData -> streams -> dataPoints/splits` and `activity`).

## 1) How FIT files are organized

A FIT file is a container of **message types**. Each message type has fields. The most important ones for this project are:

- `record`: time-series points (timestamp, distance, speed, heart rate, altitude, grade...)
- `lap`: aggregate stats for each lap
- `session`: aggregate stats for the whole activity (total distance, avg speed, avg heart rate, etc.)
- `file_id`: file/device metadata

Important: not every FIT file contains every field. Devices and recording settings differ.
So your parser must have fallbacks (for example set missing `speed`/`grade` to `0`).

In [1]:
from fitparse import FitFile
import pandas as pd
import numpy as np
from datetime import datetime

fit_path = "./data/example_ACTIVITY.fit"  # replace with the path to your own .fit file
fit = FitFile(fit_path)

In [2]:
def messages_to_df(fit_obj, message_name: str) -> pd.DataFrame:
    rows = []
    for msg in fit_obj.get_messages(message_name):
        row = {}
        for field in msg:
            row[field.name] = field.value
        if row:
            rows.append(row)
    return pd.DataFrame(rows)

record_df = messages_to_df(fit, "record")
lap_df = messages_to_df(fit, "lap")
session_df = messages_to_df(fit, "session")

print("record rows:", len(record_df))
print("lap rows:", len(lap_df))
print("session rows:", len(session_df))
print("record columns:", record_df.columns.tolist())
print("session columns:", session_df.columns.tolist())

record rows: 664
lap rows: 3
session rows: 1
record columns: ['distance', 'enhanced_altitude', 'heart_rate', 'position_lat', 'position_long', 'timestamp', 'unknown_134', 'unknown_135', 'unknown_136', 'unknown_143', 'enhanced_speed']
session columns: ['avg_cadence', 'avg_fractional_cadence', 'avg_heart_rate', 'avg_power', 'avg_speed', 'avg_stance_time', 'avg_stance_time_percent', 'avg_stroke_count', 'avg_stroke_distance', 'avg_temperature', 'enhanced_avg_speed', 'enhanced_max_speed', 'event', 'event_group', 'event_type', 'first_lap_index', 'intensity_factor', 'left_right_balance', 'max_cadence', 'max_fractional_cadence', 'max_heart_rate', 'max_power', 'max_speed', 'max_temperature', 'message_index', 'nec_lat', 'nec_long', 'normalized_power', 'num_active_lengths', 'num_laps', 'pool_length', 'pool_length_unit', 'sport', 'sport_index', 'start_position_lat', 'start_position_long', 'start_time', 'sub_sport', 'swc_lat', 'swc_long', 'swim_stroke', 'threshold_power', 'timestamp', 'total_ascent'

## 2) Build `streams.dataPoints` (15-second fixed timeline)

Target units from your docs:

- `time`: seconds from activity start
- `distance`: meters
- `speed`: km/h
- `heartrate`: bpm
- `grade`: percent
- `altitude`: meters

Rules:

- create fixed timestamps `0, 15, 30, ...`
- interpolate to those exact points
- missing `speed` and `grade` become `0`

In [3]:
def _to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")

def _interp_at_times(x: np.ndarray, y: np.ndarray, target_x: np.ndarray, default_value: float = 0.0) -> np.ndarray:
    valid = ~np.isnan(y)
    if valid.sum() == 0:
        return np.full_like(target_x, fill_value=default_value, dtype=float)
    if valid.sum() == 1:
        return np.full_like(target_x, fill_value=float(y[valid][0]), dtype=float)
    return np.interp(target_x, x[valid], y[valid])

def build_data_points(record_df: pd.DataFrame, step_seconds: int = 15) -> list[dict]:
    if record_df.empty:
        return []

    work = record_df.copy()
    work["timestamp"] = pd.to_datetime(work["timestamp"], errors="coerce")
    work = work.dropna(subset=["timestamp"]).sort_values("timestamp")
    if work.empty:
        return []

    t0 = work["timestamp"].iloc[0]
    work["time"] = (work["timestamp"] - t0).dt.total_seconds()

    distance_m = _to_numeric(work["distance"]) if "distance" in work else pd.Series(np.nan, index=work.index)
    speed_ms = _to_numeric(work["speed"]) if "speed" in work else pd.Series(np.nan, index=work.index)
    heartrate = _to_numeric(work["heart_rate"]) if "heart_rate" in work else pd.Series(np.nan, index=work.index)
    grade = _to_numeric(work["grade"]) if "grade" in work else pd.Series(np.nan, index=work.index)

    if "enhanced_altitude" in work:
        altitude_m = _to_numeric(work["enhanced_altitude"])
    elif "altitude" in work:
        altitude_m = _to_numeric(work["altitude"])
    else:
        altitude_m = pd.Series(np.nan, index=work.index)

    speed_kmh = speed_ms * 3.6

    src_t = work["time"].to_numpy(dtype=float)
    max_t = int(np.nanmax(src_t))
    target_t = np.arange(0, max_t + 1, step_seconds, dtype=float)

    out_distance = _interp_at_times(src_t, distance_m.to_numpy(dtype=float), target_t, default_value=0.0)
    out_speed = _interp_at_times(src_t, speed_kmh.to_numpy(dtype=float), target_t, default_value=0.0)
    out_hr = _interp_at_times(src_t, heartrate.to_numpy(dtype=float), target_t, default_value=0.0)
    out_grade = _interp_at_times(src_t, grade.to_numpy(dtype=float), target_t, default_value=0.0)
    out_altitude = _interp_at_times(src_t, altitude_m.to_numpy(dtype=float), target_t, default_value=0.0)

    out = []
    for i in range(len(target_t)):
        out.append({
            "time": int(target_t[i]),
            "distance": float(out_distance[i]),
            "speed": float(max(0.0, out_speed[i])),
            "heartrate": int(round(max(0.0, out_hr[i]))),
            "grade": float(out_grade[i]) if not np.isnan(out_grade[i]) else 0.0,
            "altitude": float(out_altitude[i]) if not np.isnan(out_altitude[i]) else 0.0
        })

    return out

data_points = build_data_points(record_df, step_seconds=15)
print("dataPoints count:", len(data_points))
print("first 2 points:")
print(data_points[:2])

dataPoints count: 142
first 2 points:
[{'time': 0, 'distance': 0.0, 'speed': 0.0, 'heartrate': 96, 'grade': 0.0, 'altitude': 202.79999999999995}, {'time': 15, 'distance': 0.0, 'speed': 0.0, 'heartrate': 100, 'grade': 0.0, 'altitude': 204.0}]


In [4]:
def build_splits_from_datapoints(data_points: list[dict]) -> list[dict]:
    if not data_points:
        return []

    dfp = pd.DataFrame(data_points)
    if dfp.empty or "distance" not in dfp:
        return []

    # km index starts from 1 for distances > 0
    dfp = dfp[dfp["distance"] > 0].copy()
    if dfp.empty:
        return []

    dfp["km"] = (dfp["distance"] // 1000).astype(int) + 1

    splits = []
    for km, grp in dfp.groupby("km"):
        splits.append({
            "km": int(km),
            "avgSpeed": float(grp["speed"].mean()) if "speed" in grp else 0.0,
            "avgHeartrate": int(round(grp["heartrate"].mean())) if "heartrate" in grp else 0,
            "avgGrade": float(grp["grade"].mean()) if "grade" in grp else 0.0,
            "maxSpeed": float(grp["speed"].max()) if "speed" in grp else 0.0,
            "minSpeed": float(grp["speed"].min()) if "speed" in grp else 0.0
        })

    return splits

splits = build_splits_from_datapoints(data_points)
print("splits count:", len(splits))
print("first 2 splits:")
print(splits[:2])

splits count: 14
first 2 splits:
[{'km': 1, 'avgSpeed': 0.0, 'avgHeartrate': 134, 'avgGrade': 0.0, 'maxSpeed': 0.0, 'minSpeed': 0.0}, {'km': 2, 'avgSpeed': 0.0, 'avgHeartrate': 160, 'avgGrade': 0.0, 'maxSpeed': 0.0, 'minSpeed': 0.0}]


In [5]:
def _n(value, default=0.0):
    try:
        if value is None:
            return default
        return float(value)
    except Exception:
        return default

def build_activity(session_df: pd.DataFrame, *, activity_id=123, name="Morning Ride", activity_type="Ride") -> dict:
    if session_df.empty:
        return {
            "id": activity_id,
            "name": name,
            "type": activity_type,
            "distance": 0,
            "moving_time": 0,
            "elapsed_time": 0,
            "total_elevation_gain": 0,
            "average_speed": 0,
            "max_speed": 0,
            "average_heartrate": 0,
            "max_heartrate": 0,
            "start_date": datetime.utcnow().isoformat() + "Z"
        }

    s = session_df.iloc[0]

    start_time = s.get("start_time")
    if pd.notna(start_time):
        start_date = pd.to_datetime(start_time).isoformat().replace("+00:00", "Z")
    else:
        start_date = datetime.utcnow().isoformat() + "Z"

    avg_speed_kmh = _n(s.get("avg_speed"), 0.0) * 3.6
    max_speed_kmh = _n(s.get("max_speed"), 0.0) * 3.6

    moving_time = int(_n(s.get("total_timer_time"), _n(s.get("total_elapsed_time"), 0)))
    elapsed_time = int(_n(s.get("total_elapsed_time"), 0))

    return {
        "id": activity_id,
        "name": name,
        "type": activity_type,
        "distance": int(_n(s.get("total_distance"), 0)),
        "moving_time": moving_time,
        "elapsed_time": elapsed_time,
        "total_elevation_gain": int(_n(s.get("total_ascent"), 0)),
        "average_speed": round(avg_speed_kmh, 3),
        "max_speed": round(max_speed_kmh, 3),
        "average_heartrate": int(_n(s.get("avg_heart_rate"), 0)),
        "max_heartrate": int(_n(s.get("max_heart_rate"), 0)),
        "start_date": start_date
    }

activity = build_activity(session_df)
print(activity)

{'id': 123, 'name': 'Morning Ride', 'type': 'Ride', 'distance': 13781, 'moving_time': 2055, 'elapsed_time': 2120, 'total_elevation_gain': 118, 'average_speed': 0.0, 'max_speed': 0.0, 'average_heartrate': 156, 'max_heartrate': 172, 'start_date': '2026-04-12T13:07:17'}


In [6]:
payload = {
    "workoutFocus": "Base",
    "analysisData": [
        {
            "streams": {
                "dataPoints": data_points,
                "splits": splits
            },
            "activity": activity
        }
    ]
}

print("Payload ready. Preview:")
print({
    "workoutFocus": payload["workoutFocus"],
    "dataPoints_count": len(payload["analysisData"][0]["streams"]["dataPoints"]),
    "splits_count": len(payload["analysisData"][0]["streams"]["splits"]),
    "activity": payload["analysisData"][0]["activity"]
})

Payload ready. Preview:
{'workoutFocus': 'Base', 'dataPoints_count': 142, 'splits_count': 14, 'activity': {'id': 123, 'name': 'Morning Ride', 'type': 'Ride', 'distance': 13781, 'moving_time': 2055, 'elapsed_time': 2120, 'total_elevation_gain': 118, 'average_speed': 0.0, 'max_speed': 0.0, 'average_heartrate': 156, 'max_heartrate': 172, 'start_date': '2026-04-12T13:07:17'}}


## 3) Notes for your Node.js `.ts` implementation

When you port this logic to TypeScript:

- parse `record`, `lap`, `session` messages
- compute relative `time` from first record timestamp
- resample at exactly 15-second steps
- convert speed from m/s to km/h (`* 3.6`)
- map `heart_rate` -> `heartrate`
- default missing `speed` and `grade` to `0`
- create per-km splits from interpolated datapoints
- build `activity` from `session` fields with fallbacks

If `avg_speed` is missing in session, compute fallback: `total_distance / total_elapsed_time` and convert to km/h.